# Zomato Delivery Operations Analysis Using PySpark
## Stage 8: Controlled Performance Experiment (PySpark vs. Pandas)

**Author:** Computer Engineering Mini-Project  
**Frameworks:** Apache Spark 4.2 / PySpark vs. Pandas 2.3  
**Dataset:** `zomato_cleaned.parquet` (45,584 baseline rows & 455,840 10x scaled rows)

---

### Objectives of Stage 8:
1. Conduct a scientifically fair, controlled benchmark comparing PySpark and Pandas on the exact same dataset.
2. Measure equivalent operations: GroupBy Aggregation and Multi-Condition Filtering.
3. Account for JVM warmup, task scheduling latency, and lazy evaluation.
4. Benchmark horizontal scaling on a 10x synthesized dataset (455,840 records).
5. Provide an objective Big Data Engineering interpretation without biased assumptions that distributed computing is automatically faster for small datasets.

In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from src.performance import run_benchmark

# Run controlled benchmark and display metrics table
summary_results = run_benchmark()

STAGE 8: CONTROLLED PERFORMANCE EXPERIMENT (PYSPARK VS. PANDAS)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/13 23:58:22 WARN Utils: Your hostname, Rajdeeps-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.35 instead (on interface en0)
26/10/13 23:58:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/13 23:58:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



[Dataset A] Baseline Dataset: 45,584 records



--- [Experiment 1: GroupBy & Aggregation] ---


 Pandas Execution Time:  1.55 ms
 PySpark Execution Time: 81.07 ms

--- [Experiment 2: Multi-Condition Filtering] ---


 Pandas Execution Time:  2.81 ms
 PySpark Execution Time: 90.40 ms

--- [Experiment 3: 10x Scaled Dataset (~455k rows)] ---


 Created 10x Scaled Spark DataFrame: 455,840 rows
 Created 10x Scaled Pandas DataFrame: 455,840 rows


 10x Pandas GroupBy Time:  12.20 ms
 10x PySpark GroupBy Time: 196.03 ms

PERFORMANCE BENCHMARK SUMMARY
                           Test  Pandas (ms)  PySpark (ms)                      Winner
 GroupBy Aggregation (45k rows)         1.55         81.07     Pandas (lower overhead)
        Multi-Filter (45k rows)         2.81         90.40 Pandas (in-memory C vector)
GroupBy Aggregation (455k rows)        12.20        196.03                 Competitive

Key Engineering Findings:
1. Small Dataset Tradeoff: At 45,584 records, Pandas outperforms PySpark because PySpark incurs
   JVM coordination, task scheduling, and Py4J socket communication overhead.
2. Scaling Trajectory: As data volumes exceed single-machine memory or scale beyond hundreds of megabytes,
   PySpark's distributed partitions, Catalyst optimizer, and out-of-core execution prevent Out-Of-Memory (OOM) crashes.
3. Storage Impact: Reading Parquet columnar format provides predicate pushdown and column projection,
   significantly a

### Architectural Analysis & Viva Takeaways
1. **Why Pandas is Faster on Small Datasets:** At 45,584 records (~6 MB), Pandas operates purely in contiguous RAM using highly optimized C/Cython vectorization. PySpark incurs JVM startup, Py4J gateway socket serialization, task DAG compilation, and thread pool scheduling overhead (~70 ms minimum baseline).
2. **When PySpark Becomes Essential:** When dataset size scales to tens of gigabytes or terabytes (exceeding driver RAM), Pandas crashes with an `OutOfMemoryError`. PySpark partitions the workload across worker nodes, streaming data out-of-core and executing resilient distributed computations with fault tolerance.